# MATS Simplex Take-Home: Exploratory Analysis
Non-ergodic Mess3 mixture — geometry, probing, and subspace overlap.

In [ ]:
import sys, os

%pip install "numpy<2.0" transformer_lens scikit-learn matplotlib tqdm -q

if not os.path.exists('/content/MATS-take-home'):
    !git clone https://github.com/vedantgaur/MATS-take-home.git
else:
    %cd /content/MATS-take-home && !git pull && %cd /content

sys.path.append('/content/MATS-take-home')

import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from transformer_lens import HookedTransformer

from data.mess3_generator import Mess3Process, NonErgodicMess3Dataset
from models.train import get_toy_config, train_model
from analysis.geometry import extract_activations, calculate_cev, plot_cev, plot_2d_pca, extract_all_layers
from analysis.orthogonality import compare_all_processes

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 1. Data Generation

In [ ]:
p1 = Mess3Process(alpha=0.60, x=0.15)  # Process 1
p2 = Mess3Process(alpha=0.79, x=0.11)  # Process 2 — similar dynamics to P1
p3 = Mess3Process(alpha=0.60, x=0.50)  # Process 3 — y=0, very different entropy

processes = [p1, p2, p3]
seq_length = 16

train_dataset = NonErgodicMess3Dataset(num_samples=50000, seq_length=seq_length, processes=processes)
val_dataset   = NonErgodicMess3Dataset(num_samples=2000,  seq_length=seq_length, processes=processes)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=256, shuffle=False)

print(f'Train: {len(train_dataset)} samples | Val: {len(val_dataset)} samples')
print(f'Sequence length: {seq_length} | Vocab size: 3')

## 2. Training

In [ ]:
cfg   = get_toy_config(vocab_size=3, d_model=64, n_ctx=seq_length)
model = HookedTransformer(cfg).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model params: {total_params:,}')

loss_history = train_model(model, train_loader, epochs=200, lr=1e-3)

plt.figure(figsize=(6, 4))
plt.plot(loss_history)
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Cross Entropy Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Final loss: {loss_history[-1]:.4f}')

## 3. Residual Stream Geometry (Layer-wise)

For each layer: (a) CEV to measure effective dimensionality, (b) PCA scatter colored by process ID,
(c) subspace overlap heatmap with numeric scores.

In [ ]:
print('Extracting activations across all layers...')
acts_by_layer, labels = extract_all_layers(model, val_loader)
n_layers = len(acts_by_layer)

# Store PCA models for later use
pca_models = {}

for layer_idx in range(n_layers):
    print(f'\n{"="*50}')
    print(f'LAYER {layer_idx}')
    print(f'{"="*50}')

    acts = acts_by_layer[layer_idx]

    # --- CEV ---
    pca_model, cev = calculate_cev(acts)
    pca_models[layer_idx] = pca_model
    dims_95 = np.argmax(cev >= 0.95) + 1
    print(f'  Dims for 95% variance: {dims_95}  (joint representation bound: 8)')

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # CEV plot
    axes[0].plot(range(1, len(cev)+1), cev, marker='o', markersize=3, lw=1)
    axes[0].axhline(0.95, color='r', ls='--', label='95%')
    axes[0].axvline(dims_95, color='g', ls='--', label=f'{dims_95} dims')
    axes[0].axvline(8, color='orange', ls=':', label='Theoretical (8)')
    axes[0].set_title(f'Layer {layer_idx}: CEV')
    axes[0].set_xlabel('# Principal Components')
    axes[0].set_ylabel('Cumulative Explained Variance')
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)

    # PC1 vs PC2
    proj = pca_model.transform(acts)
    sc = axes[1].scatter(proj[:, 0], proj[:, 1], c=labels, cmap='viridis', alpha=0.6, s=10)
    plt.colorbar(sc, ax=axes[1], label='Process ID')
    axes[1].set_title(f'Layer {layer_idx}: PC1 vs PC2 (macroscopic)')
    axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')

    # PC3 vs PC4
    sc2 = axes[2].scatter(proj[:, 2], proj[:, 3], c=labels, cmap='viridis', alpha=0.6, s=10)
    plt.colorbar(sc2, ax=axes[2], label='Process ID')
    axes[2].set_title(f'Layer {layer_idx}: PC3 vs PC4 (microscopic)')
    axes[2].set_xlabel('PC3'); axes[2].set_ylabel('PC4')

    plt.tight_layout()
    plt.show()

    # --- Subspace Overlap ---
    overlap_matrix = compare_all_processes(acts, labels, k_dims=2)
    process_names = ['Process 1', 'Process 2', 'Process 3']

    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(overlap_matrix, cmap='magma', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, label='Overlap (0=Orthogonal, 1=Parallel)')
    ax.set_xticks([0,1,2]); ax.set_xticklabels(process_names, rotation=20)
    ax.set_yticks([0,1,2]); ax.set_yticklabels(process_names)
    ax.set_title(f'Layer {layer_idx}: Subspace Overlap')

    # Annotate with numeric values
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f'{overlap_matrix[i,j]:.3f}',
                    ha='center', va='center',
                    color='white' if overlap_matrix[i,j] < 0.6 else 'black',
                    fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f'  Overlap matrix (numeric):')
    print(f'    P1-P2: {overlap_matrix[0,1]:.4f}')
    print(f'    P1-P3: {overlap_matrix[0,2]:.4f}')
    print(f'    P2-P3: {overlap_matrix[1,2]:.4f}')

## 4. Context Position Evolution

Project activations at different sequence positions onto the last-layer PCA basis.
Uses the **full** validation set (not just one batch) for reliable geometry.

In [ ]:
def extract_all_positions(model, dataloader):
    """Extract residual stream activations at ALL sequence positions."""
    model.eval()
    last_layer = model.cfg.n_layers - 1
    hook_name  = f'blocks.{last_layer}.hook_resid_post'

    all_acts   = []   # will be (N, seq_len, d_model)
    all_labels = []

    with torch.no_grad():
        for batch_x, _, labs in dataloader:
            batch_x = batch_x.to(model.cfg.device)
            _, cache = model.run_with_cache(batch_x, names_filter=[hook_name])
            all_acts.append(cache[hook_name].cpu().numpy())
            all_labels.append(labs.numpy())

    return np.concatenate(all_acts, axis=0), np.concatenate(all_labels)


print('Extracting full-sequence activations from val set...')
all_pos_acts, pos_labels = extract_all_positions(model, val_loader)
print(f'Shape: {all_pos_acts.shape}  (samples, seq_len, d_model)')

# Use the last-layer PCA (fit on last-position acts) as the fixed projection basis
final_pca = pca_models[n_layers - 1]

positions_to_plot = [0, 3, 7, 14]  # l = 1, 4, 8, 15
fig, axes = plt.subplots(1, len(positions_to_plot), figsize=(20, 5), sharex=True, sharey=True)

for i, pos in enumerate(positions_to_plot):
    pos_acts = all_pos_acts[:, pos, :]       # (N, d_model)
    proj     = final_pca.transform(pos_acts) # project onto shared PCA basis

    sc = axes[i].scatter(proj[:, 0], proj[:, 1], c=pos_labels,
                         cmap='viridis', alpha=0.7, s=12)
    axes[i].set_title(f'Position l={pos+1}')
    axes[i].set_xlabel('PC 1')
    if i == 0:
        axes[i].set_ylabel('PC 2')

plt.suptitle('Activation Geometry Evolution Across Context Window', fontsize=14)
plt.colorbar(sc, ax=axes[-1], label='Process ID')
plt.tight_layout()
plt.show()

## 5. Linear Probe: Residual Stream → Belief States

The model doesn't know which process generated a sequence, so for optimal prediction it must
maintain belief trajectories under ALL process hypotheses simultaneously:
P(x_{l+1} | x_{1:l}) = Σ_k P(k | x_{1:l}) · P(x_{l+1} | x_{1:l}, k).

We compute η_k(x_{1:l}) for every k by running the forward algorithm under each process's
dynamics on the observed tokens. We also compute the Bayesian posterior P(k | x_{1:l})
and probe for process classification accuracy.

In [ ]:
def compute_belief_trajectory(token_sequence, process: Mess3Process):
    """
    Run the HMM forward algorithm under a given process's dynamics.
    Returns (trajectory, log_likelihood):
      trajectory: array of shape (seq_len, 3)
      log_likelihood: scalar log P(sequence | process)
    """
    T_matrices = [process.T0, process.T1, process.T2]
    eta = process.state_dist.copy()
    trajectory = []
    log_lik = 0.0

    for token in token_sequence:
        T_x = T_matrices[int(token)]
        unnorm = eta @ T_x
        norm_const = unnorm.sum()  # P(x_t | x_{1:t-1}, process)
        log_lik += np.log(norm_const + 1e-30)
        eta = unnorm / norm_const
        trajectory.append(eta.copy())

    return np.array(trajectory), log_lik


def build_belief_targets(val_dataset, processes):
    """
    For each sample, compute belief trajectories under ALL process hypotheses
    at the last position. The model must maintain beliefs for all processes
    to predict optimally: P(x_{l+1}|x_{1:l}) = sum_k P(k|x_{1:l}) P(x_{l+1}|x_{1:l},k).

    Returns:
      beliefs: (N, 9) — [η_1(x), η_2(x), η_3(x)] for all k
      posteriors: (N, 3) — Bayesian posterior P(process=k | x_{1:l})
      true_labels: (N,) — ground-truth process labels
    """
    n_proc = len(processes)
    all_beliefs = []
    all_posteriors = []
    all_labels = []

    for idx in range(len(val_dataset)):
        x, _, label = val_dataset[idx]
        tokens = x.numpy()

        beliefs_k = []
        log_lik_k = []

        for k in range(n_proc):
            traj, ll = compute_belief_trajectory(tokens, processes[k])
            beliefs_k.append(traj[-1])  # belief at last position under process k
            log_lik_k.append(ll)

        # Bayesian posterior with uniform prior
        log_lik_k = np.array(log_lik_k)
        log_lik_k -= log_lik_k.max()
        posterior = np.exp(log_lik_k)
        posterior /= posterior.sum()

        all_beliefs.append(np.concatenate(beliefs_k))  # (9,)
        all_posteriors.append(posterior)                 # (3,)
        all_labels.append(int(label))

    return np.array(all_beliefs), np.array(all_posteriors), np.array(all_labels)


print('Computing ground-truth belief states under all process hypotheses...')
belief_targets, process_posteriors, true_labels = build_belief_targets(val_dataset, processes)
print(f'Belief target shape: {belief_targets.shape}  (samples, 9)')
print(f'Process posterior shape: {process_posteriors.shape}  (samples, 3)')
print(f'Mean Bayes-optimal posterior confidence: {process_posteriors.max(axis=1).mean():.3f}')
print(f'Sample all-process beliefs: {belief_targets[0].round(3)}')

In [ ]:
from sklearn.linear_model import Ridge, RidgeCV, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

acts_final = acts_by_layer[n_layers - 1]   # (N, 64)

idx = np.arange(len(acts_final))
idx_train, idx_test = train_test_split(idx, test_size=0.2, random_state=42)

# --- Belief State Probe (all-process hypotheses) ---
probe = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=5)
probe.fit(acts_final[idx_train], belief_targets[idx_train])
y_pred = probe.predict(acts_final[idx_test])
r2_overall = r2_score(belief_targets[idx_test], y_pred)

r2_per_process = []
for k in range(len(processes)):
    start, end = k*3, k*3 + 3
    r2_k = r2_score(belief_targets[idx_test, start:end], y_pred[:, start:end])
    r2_per_process.append(r2_k)

print(f'=== Belief State Probe (all-process hypotheses) ===')
print(f'  Overall R²:    {r2_overall:.4f}')
for k, r2_k in enumerate(r2_per_process):
    print(f'  Process {k+1} belief R²: {r2_k:.4f}')
print(f'  Best Ridge alpha: {probe.alpha_}')

# --- Process Classification Probe ---
clf = LogisticRegression(max_iter=1000, C=1.0)
clf.fit(acts_final[idx_train], true_labels[idx_train])
clf_acc = clf.score(acts_final[idx_test], true_labels[idx_test])

print(f'\n=== Process Classification Probe ===')
print(f'  Accuracy: {clf_acc:.4f}  (chance = 0.333)')

# --- Process Posterior Probe ---
post_probe = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=5)
post_probe.fit(acts_final[idx_train], process_posteriors[idx_train])
post_pred = post_probe.predict(acts_final[idx_test])
r2_posterior = r2_score(process_posteriors[idx_test], post_pred)
print(f'  Posterior R²:  {r2_posterior:.4f}')

# --- Random baseline ---
random_acts = np.random.randn(*acts_final.shape)
random_probe = Ridge(alpha=1.0).fit(random_acts[idx_train], belief_targets[idx_train])
r2_random = r2_score(belief_targets[idx_test], random_probe.predict(random_acts[idx_test]))
print(f'\n  Random baseline belief R²: {r2_random:.4f}')

## 6. Per-Layer Probing: Routing vs. Belief Tracking

To test whether early layers perform macroscopic routing while later layers refine belief tracking,
we train separate probes at each layer for (a) process classification and (b) belief state regression.

In [ ]:
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

idx = np.arange(len(labels))
idx_train, idx_test = train_test_split(idx, test_size=0.2, random_state=42)

layer_routing_acc = []
layer_belief_r2 = []

for layer_idx in range(n_layers):
    acts = acts_by_layer[layer_idx]

    clf = LogisticRegression(max_iter=1000, C=1.0)
    clf.fit(acts[idx_train], labels[idx_train])
    acc = clf.score(acts[idx_test], labels[idx_test])
    layer_routing_acc.append(acc)

    probe = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=5)
    probe.fit(acts[idx_train], belief_targets[idx_train])
    r2 = r2_score(belief_targets[idx_test], probe.predict(acts[idx_test]))
    layer_belief_r2.append(r2)

    print(f'Layer {layer_idx}: Routing Acc = {acc:.4f} | Belief R² = {r2:.4f}')

fig, ax1 = plt.subplots(figsize=(8, 5))
ax2 = ax1.twinx()

x_pos = np.arange(n_layers)
ax1.bar(x_pos - 0.15, layer_routing_acc, 0.3, label='Routing Accuracy', color='steelblue', alpha=0.8)
ax2.bar(x_pos + 0.15, layer_belief_r2, 0.3, label='Belief R²', color='coral', alpha=0.8)

ax1.set_xlabel('Layer')
ax1.set_ylabel('Classification Accuracy', color='steelblue')
ax2.set_ylabel('Belief R²', color='coral')
ax1.set_xticks(range(n_layers))
ax1.axhline(1/3, color='steelblue', ls='--', alpha=0.5)
ax1.set_ylim(0, 1.05)

lines1, labels1_ = ax1.get_legend_handles_labels()
lines2, labels2_ = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1_ + labels2_, loc='upper left')

plt.title('Per-Layer Probing: Routing vs. Belief Tracking')
plt.tight_layout()
plt.show()

## 7. Overlap Evolution Summary

Plot all pairwise overlap scores across layers side by side for easy comparison.

In [ ]:
overlap_by_layer = {}
for layer_idx in range(n_layers):
    acts = acts_by_layer[layer_idx]
    overlap_by_layer[layer_idx] = compare_all_processes(acts, labels, k_dims=2)

pairs = [('P1-P2', 0, 1), ('P1-P3', 0, 2), ('P2-P3', 1, 2)]
layer_ids = list(range(n_layers))

plt.figure(figsize=(8, 4))
for name, i, j in pairs:
    scores = [overlap_by_layer[l][i, j] for l in layer_ids]
    plt.plot(layer_ids, scores, marker='o', linewidth=2, label=name)

plt.xlabel('Layer')
plt.ylabel('Subspace Overlap (0=Orthogonal)')
plt.title('Pairwise Subspace Overlap Across Layers')
plt.xticks(layer_ids)
plt.ylim(0, 1)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\nNumerical overlap scores by layer:')
print(f'{"Layer":<8}', '  '.join(f'{n:<10}' for n, *_ in pairs))
for l in layer_ids:
    row = [overlap_by_layer[l][i, j] for _, i, j in pairs]
    print(f'{l:<8}', '  '.join(f'{v:<10.4f}' for v in row))